# Brussels Traffic Data Processing Pipeline

In [27]:
%reset -f

In [28]:
# Standard imports
import sys
sys.path.insert(0, '/home/ubuntu/prem')

import pandas as pd
import numpy as np
import gc
from tqdm import tqdm
import geopandas as gpd
from shapely import wkt

In [29]:
# Modular imports
from utils import log_memory, log_df_memory, load_data, save_detection_results
from filters.preprocessing import (
    filter_by_lifetime,
    attach_zones_to_objects,
    apply_footpath_zone_filter,
    compute_polygon_orientation,
    filter_parallel_vehicles,
    filter_static_objects
)
from regions.brussels.zones import get_lane_zones, get_footpath_zones, get_crosswalk_zones
from ssm.utils import load_config, assign_zones_to_vehicles
from ssm.m_drac import ModifiedDRAC
from ssm.spf import SafetyPotentialField

In [30]:
# Configuration
START_DATE = "2025-06-14"
END_DATE = "2025-06-14"
DATA_DIR = "/home/ubuntu/data/uploads/objects/clean"
OUTPUT_DIR = "/home/ubuntu/prem/results"

config = load_config("/home/ubuntu/prem/config.yaml")

print("="*70)
print("BRUSSELS TRAFFIC ANALYSIS")
print("="*70)
print(f"Date: {START_DATE} to {END_DATE}")
print("="*70)

BRUSSELS TRAFFIC ANALYSIS
Date: 2025-06-14 to 2025-06-14


## 1. Data Loading

In [31]:
print("\nLoading data...")
log_memory("Before loading")

df = load_data(DATA_DIR, START_DATE, END_DATE, dtypes=config['data']['dtypes'])

log_df_memory(df, "Loaded data")
print(f"Loaded {len(df):,} records")
df.reset_index(drop=True, inplace=True)


Loading data...
[MEMORY] Before loading: 4178.8 MB


Loading data: 100%|██████████| 336/336 [00:03<00:00, 105.35it/s] 


[DF MEMORY] Loaded data: 2254.9 MB (38,761,175 rows)
Loaded 38,761,175 records


## 2. Lifetime Filtering

In [32]:
print("\n" + "="*70)
print("Lifetime Filtering")
print("="*70)

df = filter_by_lifetime(df, config['preprocessing']['lifetime_filter']['min_lifespan'])
log_df_memory(df, "After lifetime filter")


Lifetime Filtering
[lifespan filter] Removed 11,535 short-lived IDs
  Before: 81,847 IDs (38,761,175 rows)
  After: 70,312 IDs (38,198,691 rows)
[DF MEMORY] After lifetime filter: 2513.6 MB (38,198,691 rows)


np.float64(2513.6086263656616)

## 3. Footpath Zone Filtering

In [33]:
print("\n" + "="*70)
print("Footpath Zone Filtering")
print("="*70)

footpath_zones = get_footpath_zones()
zones_df = pd.DataFrame(footpath_zones)
zones_df["geometry"] = zones_df["vertices"].apply(wkt.loads)
gdf_zones = gpd.GeoDataFrame(zones_df, geometry="geometry")

# CRITICAL: Call attach_zones_to_objects ONCE
print(f"Attaching footpath zones to {len(df):,} rows...")
log_memory("Before footpath zones")

df = attach_zones_to_objects(df, gdf_zones, how="left", batch_size=100000)

log_memory("After footpath zones")
print(f"✓ Zones attached! Total rows: {len(df):,}")

df = apply_footpath_zone_filter(df)
df = df.drop(columns=['zone'], errors='ignore')
gc.collect()
log_memory("After footpath filter")


Footpath Zone Filtering
Attaching footpath zones to 38,198,691 rows...
[MEMORY] Before footpath zones: 6655.7 MB


Zone assignment batches: 100%|██████████| 382/382 [00:53<00:00,  7.11it/s]


[MEMORY] After footpath zones: 6505.1 MB
✓ Zones attached! Total rows: 38,198,691
[footpath zone] Removed 828 objects
[MEMORY] After footpath filter: 6459.3 MB


6459.3046875

## 4. Crosswalk Zone Filtering

In [34]:
print("\n" + "="*70)
print("Crosswalk Zone Filtering")
print("="*70)

crosswalk_zones = get_crosswalk_zones()
zones_df = pd.DataFrame(crosswalk_zones)
zones_df["geometry"] = zones_df["vertices"].apply(wkt.loads)
gdf_zones = gpd.GeoDataFrame(zones_df, geometry="geometry")
gdf_zones["orientation_deg"] = gdf_zones["geometry"].apply(compute_polygon_orientation)

print(f"Attaching crosswalk zones to {len(df):,} rows...")
log_memory("Before crosswalk zones")

df = attach_zones_to_objects(df, gdf_zones, how="left", batch_size=100000)

log_memory("After crosswalk zones")
print(f"✓ Zones attached! Total rows: {len(df):,}")

# Filter parallel vehicles
removed_ids_global = []
df_in_zones = df[df['zone'].notnull()].copy()

for zone_id in df_in_zones['zone'].unique():
    df_zone = df_in_zones[df_in_zones['zone'] == zone_id]
    orientation = gdf_zones[gdf_zones['id'] == zone_id]['orientation_deg'].iloc[0]
    parallel_ids, _ = filter_parallel_vehicles(df_zone, orientation, threshold=4.0)
    removed_ids_global.extend(parallel_ids)

df = df[~df['id'].isin(removed_ids_global)]
print(f"[crosswalk] Removed {len(removed_ids_global):,} parallel vehicles")

df = df.drop(columns=['zone'], errors='ignore')
gc.collect()
log_memory("After crosswalk filter")


Crosswalk Zone Filtering
Attaching crosswalk zones to 37,447,575 rows...
[MEMORY] Before crosswalk zones: 6459.3 MB


Zone assignment batches: 100%|██████████| 375/375 [00:51<00:00,  7.27it/s]


[MEMORY] After crosswalk zones: 6465.7 MB
✓ Zones attached! Total rows: 37,447,575
[crosswalk] Removed 7 parallel vehicles
[MEMORY] After crosswalk filter: 6464.7 MB


6464.67578125

## 5. Static Object Removal

In [35]:
print("\n" + "="*70)
print("Static Object Removal")
print("="*70)

df = filter_static_objects(df, 
    static_threshold=config['preprocessing']['static_filter']['min_speed'],
    static_ratio_min=0.8)

log_df_memory(df, "After static filter")


Static Object Removal
[static filter] Found 9,132 static objects
[static filter] Removed 9,132 static objects
  Before: 69,477 IDs (37,430,780 rows)
  After: 60,345 IDs (17,176,653 rows)
[DF MEMORY] After static filter: 1130.3 MB (17,176,653 rows)


np.float64(1130.284363746643)

## 6. Lane Zone Assignment

In [36]:
print("\n" + "="*70)
print("Lane Zone Assignment")
print("="*70)

lane_zones = get_lane_zones()
df = assign_zones_to_vehicles(df, lane_zones)

print(df['zone'].value_counts())
print(f"\nVehicles in lanes: {(df['zone'] != 'unknown').sum():,}")

df_lanes = df[df['zone'] != 'unknown'].copy()
log_df_memory(df_lanes, "Lane vehicles")


Lane Zone Assignment

Assigning zones to 17,176,653 vehicles using spatial join...
Processing in batches of 100,000 rows


Zone assignment: 100%|██████████| 172/172 [00:31<00:00,  5.41it/s]



✓ Zone assignment complete!
  Vehicles in zones: 5,261,175
  Vehicles outside zones: 12,216,553
zone
unknown    12216553
E-L1        2058116
E-L2         862872
B-L2         825956
C-L1         582323
B-L1         301075
C-L2         284541
D-L1         267932
D-L2          47276
A-L1          31084
Name: count, dtype: int64

Vehicles in lanes: 5,261,175
[DF MEMORY] Lane vehicles: 652.3 MB (5,261,175 rows)


np.float64(652.2681713104248)

## 7. M-DRAC Detection

In [37]:
print("\n" + "="*70)
print("M-DRAC Detection")
print("="*70)

# Generate base pairs first (EXACT OLD CODE WORKFLOW)
from ssm.utils import find_all_nearby_pairs, get_mdrac_pairs

print("\nGenerating nearby pairs...")
log_memory("Before pair generation")

# OLD code signature: find_all_nearby_pairs(df, config)
base_pairs = find_all_nearby_pairs(df_lanes, config)

print(f"✓ Generated {len(base_pairs):,} base pairs")
log_memory("After pair generation")

# Filter pairs for M-DRAC 
print("\nFiltering pairs for M-DRAC...")
mdrac_pairs = get_mdrac_pairs(base_pairs, config, skip_pair_generation=True)
print(f"✓ M-DRAC pairs after filtering: {len(mdrac_pairs):,}\"")

# Detect conflicts from filtered pairs
print("\nDetecting M-DRAC conflicts...")
mdrac_detector = ModifiedDRAC(config)
mdrac_conflicts = mdrac_detector.detect(mdrac_pairs, is_pairs_data=True)

print(f"\n{'='*70}")
print(f"M-DRAC Conflicts: {len(mdrac_conflicts):,}")
print(f"{'='*70}")



M-DRAC Detection

Generating nearby pairs...
[MEMORY] Before pair generation: 5737.4 MB
  Filtered vehicles: 2,153,630
  Generating pairs (max_distance=8.0m)...
  Processing 643,949 timestamps (batch_size=5,000)


  Pair generation: 100%|██████████| 129/129 [00:04<00:00, 29.11it/s]


  Applying overlap filter (buffer=0.5m)...
  ✓ Generated 119,431 nearby pairs (after overlap filter)
  ✓ Added zone information (zone1/zone2 columns)
✓ Generated 119,431 base pairs
[MEMORY] After pair generation: 5737.7 MB

Filtering pairs for M-DRAC...
 Approaching pairs: 56,284
  Zone filter (same lane): 25,905 pairs (filtered 30,379 different-lane)
  Lateral filter (<= 3.0m): 15,556 pairs (filtered 10,349 not aligned)
  ✓ Total filtered: 40,728 pairs | Remaining: 15,556 pairs
  Speed diff > 0.5: 8,008
  Final MDRAC pairs: 280
✓ M-DRAC pairs after filtering: 280"

Detecting M-DRAC conflicts...
 Approaching pairs: 280
  Zone filter (same lane): 280 pairs (filtered 0 different-lane)
  Lateral filter (<= 3.0m): 280 pairs (filtered 0 not aligned)
  ✓ Total filtered: 0 pairs | Remaining: 280 pairs
  Speed diff > 0.5: 280
  Final MDRAC pairs: 280

M-DRAC Conflicts: 3


In [38]:
# Save M-DRAC results
if len(mdrac_conflicts) > 0:
    mdrac_path = save_detection_results(mdrac_conflicts, OUTPUT_DIR, 'mdrac', 'brussels', START_DATE)
    print(f"Saved to {mdrac_path}")

✓ Saved 3 conflicts to /home/ubuntu/prem/results/brussels/mdrac/14/mdrac_14.csv
Saved to /home/ubuntu/prem/results/brussels/mdrac/14/mdrac_14.csv


## Complete

In [39]:
print("\n" + "="*70)
print("BRUSSELS ANALYSIS COMPLETE")
print("="*70)
print(f"M-DRAC: {len(mdrac_conflicts):,}")
print("="*70)


BRUSSELS ANALYSIS COMPLETE
M-DRAC: 3
